In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# from langchain_community.document_loaders import TextLoader,PyPDFLoader

from langchain_core.runnables import RunnableLambda,RunnableParallel
from langchain.tools import tool
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated

from langchain_community.tools import DuckDuckGoSearchRun,ShellTool
from langchain_core.tools import InjectedToolArg

from langchain_community.document_loaders import TextLoader,PyPDFLoader,Docx2txtLoader,WebBaseLoader

C:\Users\Mamatha-Win10\AppData\Local\Temp\ipykernel_2212\688021366.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun,ShellTool
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
''' Resume anaylser pdf(format), word(format), (protofilo link of a person)'''

llm= ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.getenv('GROQ_API_KEY'),
    base_url= "https://api.groq.com/openai/v1",
    max_tokens= 2000
)


In [4]:
result=llm.invoke("who is Dr.Rajkumar").content
print(result)

**Dr. Rajkumar** is most widely recognized as the legendary figure of Kannada cinema, but the name also belongs to a few other notable people. Here’s a quick rundown of the most prominent ones:

| Name | Field | Why they’re well‑known |
|------|-------|------------------------|
| **Dr. Rajkumar (1929 – 2006)** | Film & Music (Kannada) | Real name **Singanalluru Puttaswamaiah Muthuraj**, he was a beloved actor, playback singer, and cultural icon in the state of Karnataka. He starred in more than 200 movies, earned the title *“Nata Saarvabhouma”* (Emperor of Actors), and received honors such as the Padma Bhushan (1991) and the Dadasaheb Phalke Award (1999). |
| **Dr. Rajkumar (politician)** | Politics (India) | A member of the **Bharatiya Janata Party (BJP)** from Karnataka, serving as a Member of Parliament (Lok Sabha) and holding various ministerial portfolios in the state government. |
| **Dr. Rajkumar (neurosurgeon)** | Medicine | **Dr. Rajkumar N. K.**, a noted Indian neurosurgeon a

In [5]:
import tiktoken

def truncate_text(text: str, max_tokens: int = 6000) -> str:
    """Trim text so it fits comfortably within the model's context."""
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return enc.decode(tokens[:max_tokens])

In [6]:
def load_document(source:str)->str:
    '''Takes a file path or UR and returns the extraceted text'''

    source_lower= source.lower()
    print(source_lower)

    if source_lower.startswith(("http://","https://")):
        loader= WebBaseLoader(source)
    elif source_lower.endswith(".pdf"):
        loader= PyPDFLoader(source)
    elif source_lower.endswith(".docx"):
        loader= Docx2txtLoader(source)
    else:
        raise ValueError(f"UnSupported format: {source}")
    
    docs= loader.load()
    full_text="\n\n".join(d.page_content for d in docs)
    return truncate_text(full_text)

In [7]:
data=load_document("Mamatha Resume word am.docx")


mamatha resume word am.docx


In [19]:
resume_score_prompt= PromptTemplate(
    template="""You are an expert resume reviwer.
    **Score**: X/100
    **Overall Match**: X%
    Keywords matched: • ...
    Missing keywords: • ...
    Readability Score: X/100
    ATS Compatibility Score: X/100
    
    2-liner summary: ...
    Skill gap analysis: • ...
    Overall improvement suggestions: • ...
    Industry specific feedback: • ...
    Job: {job_description}
    Resume: {resume_text}
    Be honest, use rubrics
    """,
    input_variables=["resume_text","job_description"]
)


In [34]:
parser= StrOutputParser()
def run_chain(prompt,**kwargs):
    return (prompt | llm | parser).invoke(kwargs)

In [35]:

RESUME_CHECKER_PROMPT = PromptTemplate.from_template("""
Score resume standalone (clarity, format, ATS, skills): EXACT structure:
1. **Score**: X/100
2. **Strengths**: • ...
3. **Weaknesses**: • ...
4. **Skills Mentioned**: • ...
5. **Recommended Skills**: • ...
6. **Next Career Steps**: • ...
Resume: {context}
""")

In [36]:
# job-specific review
run_chain(resume_score_prompt, resume_text=data, job_description="frontend developer")

# standalone review
run_chain(RESUME_CHECKER_PROMPT, context=data)

# # any future prompt you add
# run_chain(your_new_prompt, some_variable=value)


'**1. Score:** **45/100**\n\n---\n\n**2. Strengths**  \n- Complete contact information (phone, email, address).  \n- Demonstrates a range of work experience across admin, HR recruiting, and admissions management.  \n- Lists basic computer literacy (Windows OS, Tally ERP\u202f9, MS Office).  \n- Includes language proficiency (English, Kannada, Hindi).  \n- Shows personal traits such as communication, teamwork, and time‑management.\n\n---\n\n**3. Weaknesses**  \n- **Formatting & Layout:** Inconsistent headings, irregular spacing, lack of bullet points, and a mixture of tabs & spaces make the document hard to scan.  \n- **Objective Statement:** Too generic, contains spelling/grammar errors (“carrier” → “career”).  \n- **Education Section:** Missing graduation years for BSc, no degree titles, and percentages without context.  \n- **Professional Experience:** No quantifiable achievements (e.g., “processed\u202f200+ admissions per month”), no action verbs leading each bullet, dates are not u

In [37]:
from langchain.tools import tool
@tool 
def analyze_resume(source:str)->str:
    """ Anaylze a resume or portfolio and return feedback.
    
    Use this when the user provides a resume or portfolio as a
    file path (.pdf or .docx) or as a URL. Returns a structured
    review with strengths, gaps, and recommendations.
    
    """
    resume_text=load_document(source)
    return  run_chain(resume_score_prompt, resume_text=resume_text, job_description="front desk executive")

@tool
def resume_checker(source:str)->str:
    """Check a resume for clarity, format, ATS compatibility and skills.
    
    Use this when the user wants a general resume review WITHOUT a specific
    job description. Returns scores, strengths, weaknesses and career steps.
    """
    resume_text = load_document(source)
    return run_chain(RESUME_CHECKER_PROMPT, context=resume_text)


In [38]:
print(analyze_resume.name)

analyze_resume


In [39]:
print(resume_checker.description)

Check a resume for clarity, format, ATS compatibility and skills.

    Use this when the user wants a general resume review WITHOUT a specific
    job description. Returns scores, strengths, weaknesses and career steps.


In [40]:
print(analyze_resume.args)

{'source': {'title': 'Source', 'type': 'string'}}


In [41]:
from langchain.agents import create_agent

In [42]:
graph = create_agent(
    model=llm,           # your ChatOpenAI instance
    tools=[analyze_resume,resume_checker],
    system_prompt="You are a helpful assistant"
)



In [43]:
result = graph.invoke({"messages": [{"role": "user", "content": "Kindly analyze the resume at: Mamatha Resume word am.docx"}]})
print(result["messages"][-1].content)

mamatha resume word am.docx
**Resume Review – Front Desk Executive (Front‑Office / Reception)  
Candidate: Mamatha H**

---

### 1️⃣ Overall Scores (out of 100)

| Metric | Score | Comment |
|--------|-------|---------|
| **Overall Match** | **38 %** | Core front‑desk duties and industry‑specific keywords are largely missing. |
| **Readability** | **68 %** | Information is present but formatting, bullet consistency, and section headings need work. |
| **ATS Compatibility** | **44 %** | Sparse use of keywords, unconventional headings, and a large block of text reduce parsability. |
| **Composite** | **50 / 100** | Shows solid administrative experience, but not yet positioned for a front‑desk role. |

---

### 2️⃣ Keywords – Matched vs. Missing  

| **Matched Keywords** (present in resume) | **Missing / Critical Keywords** (common in Front‑Desk ads) |
|-------------------------------------------|-------------------------------------------------------------|
| Communication skills        